## Introduction<a id='introduction'></a>

**Purpose:**

Preprocess data for modeling.

**Context:**

&emsp; Pickleball is an addictively fun racquet sport combining aspects of tennis, table tennis, and badminton and pickleball players are similarly diverse with players of all ages and all sports backgrounds. Players often spend hours on the courts both playing and socializing between games. A common topic of discussion amongst players is strategy with questions such as

* "Should I be dropping or driving my third shots?"
* "What kind of shots are more common at higher levels of play?"

This project aims to answer common questions in pickleball using data from pklmart.

**Data:**

Collection of pklmart data on Kaggle (https://www.kaggle.com/datasets/cakesofspan/pklmarts-competitive-pickleball-extracts). \
Pklmart (https://pklmart.com/)

**Outcomes from this notebook (Preprocessing):**

**Outcomes from previous notebook (EDA):**


## Contents<a id='contents'></a>
* [Introduction](#introduction)
* [Contents](#contents)
* [Objectives](#objectives)
* [Preprocessing Data](#preprocessing_data)

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Andy's path to project data folder
data_path = '/content/drive/MyDrive/Colab Notebooks/DSCI 521 Final Project (Pickleball Analytics)/data/'

plt.rcParams.update({'font.size': 16})

In [58]:
game_df = pd.read_csv(data_path+'raw/game.csv')
player_df = pd.read_csv(data_path+'raw/player.csv')
team_df = pd.read_csv(data_path+'raw/team.csv')
rally_df = pd.read_csv(data_path+'raw/rally.csv')
shot_df = pd.read_csv(data_path+'raw/shot.csv')

shot_type_ref = pd.read_csv(data_path+'raw/shot_type_ref.csv', index_col=0)

shot_rally_df = pd.read_csv(data_path+'shot_rally.csv', index_col=0)

## Objectives <a id='objectives'></a>

1. Preprocess

In [5]:
rally_df.head(3)

,rally_id,game_id,match_id,rally_nbr,w_team_id,srv_team_id,srv_player_id,rtrn_team_id,rtrn_player_id,ts_player_id,...,ending_type,ending_player_id,srv_switch_ind,rtrn_switch_ind,srv_team_flipped_ind,rtrn_team_flipped_ind,srv_team_rs_player_id,srv_team_ls_player_id,rtrn_team_rs_player_id,rtrn_team_ls_player_id
0,R8968,G208,M113,6,T152,T152,P202,T155,P70,P202,...,Other,NaN,Y,N,N,N,P203,P202,P70,P205
1,R8963,G208,M113,1,T155,T152,P202,T155,P70,NaN,...,Error,P203,Y,N,N,N,P203,P202,P70,P205
2,R8964,G208,M113,2,T152,T155,P70,T152,P202,P205,...,Error,P205,N,Y,N,N,P70,P205,P203,P202


In [21]:
def get_xy_shot_seq(rally_id, shot_df):
    '''
    For given rally_id, pulls sequence of (x,y) coordinates of shots.
    Inputs:
        rally_id: identifier for rally; str
        shot_df: dataframe containing shot data; pd.DataFrame
    Outputs:
        shot_seq: array of (x, y) coordinates; np.array
    '''
    if rally_id not in shot_df.rally_id.unique():
        print('Invalid rally_id')
        return

    my_df = shot_df[shot_df['rally_id'] == rally_id][['shot_nbr', 'loc_x', 'loc_y']]
    my_df = my_df.sort_values(by='shot_nbr')

    return my_df[['loc_x', 'loc_y']].values

In [26]:
def get_shot_type_seq(rally_id, shot_df):
    '''
    For given rally_id, pulls sequence of shot types.
    Inputs:
        rally_id: identifier for rally; str
        shot_df: dataframe containing shot data; pd.DataFrame
    Ouputs:
        shot_type_seq: array of shot types; np.array
    '''
    if rally_id not in shot_df.rally_id.unique():
        print('Invalid rally_id')
        return

    my_df = shot_df[shot_df['rally_id'] == rally_id][['shot_nbr', 'shot_type']]
    my_df = my_df.sort_values(by='shot_nbr')

    return my_df['shot_type'].values

In [44]:
def get_dist_seq(rally_id, shot_df):
    '''
    For given rally_id, pulls sequence of shot distances.
    Inputs:
        rally_id: identifier for rally; str
        shot_df: dataframe containing shot data; pd.DataFrame
    Outputs:
        dist_seq: array of distance traveled for shot; np.array
    '''
    if rally_id not in shot_df.rally_id.unique():
        print('Invalid rally_id')
        return

    cols = ['shot_nbr', 'loc_x', 'loc_y', 'next_loc_x', 'next_loc_y']
    my_df = shot_df[shot_df['rally_id'] == rally_id][cols]
    my_df = my_df.sort_values(by='shot_nbr')

    delta_x2 = (my_df['next_loc_x'] - my_df['loc_x'])**2
    delta_y2 = (my_df['next_loc_y'] - my_df['loc_y'])**2
    dist_seq = np.sqrt(delta_x2 + delta_y2)

    return dist_seq

In [45]:
def get_angle_seq(rally_id, shot_df):
    '''
    For given rally_id, pulls sequence of shot angles.
    Inputs:
        rally_id: identifier for rally; str
        shot_df: dataframe containing shot data; pd.DataFrame
    Outputs:
        angle_seq: array of shot angles; np.array
    '''
    if rally_id not in shot_df.rally_id.unique():
        print('Invalid rally_id')
        return

    cols = ['shot_nbr', 'loc_x', 'loc_y', 'next_loc_x', 'next_loc_y']
    my_df = shot_df[shot_df['rally_id'] == rally_id][cols]
    my_df = my_df.sort_values(by='shot_nbr')

    delta_x = (my_df['next_loc_x'] - my_df['loc_x'])
    delta_y = (my_df['next_loc_y'] - my_df['loc_y'])
    angle_seq = np.arctan(delta_x/delta_y)/(np.pi/2)*90

    return angle_seq

In [ ]:
def get_first_serve(rally_id, rally_df):

In [69]:
game_id = 'G614'
my_df = rally_df[rally_df['game_id'] == game_id][['rally_nbr', 'srv_switch_ind', 'rtrn_switch_ind']].sort_values('rally_nbr')
my_df[10:40]

,rally_nbr,srv_switch_ind,rtrn_switch_ind
20319,11,N,N
20314,12,N,N
20332,13,N,N
20323,14,N,N
20324,15,N,N
20333,16,N,N
20334,17,N,N
20335,18,N,N
20336,19,N,N
20337,20,N,N


In [61]:
rally_df[['srv_switch_ind', 'rtrn_switch_ind']].head()

,srv_switch_ind,rtrn_switch_ind
0,Y,N
1,Y,N
2,N,Y
3,N,Y
4,Y,Y


In [56]:
match_df.columns

NameError: name 'match_df' is not defined

In [34]:
rally_id = 'R26079'
xy = shot_rally_df[shot_rally_df['rally_id'] == rally_id][['shot_nbr', 'loc_x', 'loc_y', 'next_loc_x', 'next_loc_y']]

In [35]:
xy

,shot_nbr,loc_x,loc_y,next_loc_x,next_loc_y
109210,4,3.93,6.25,14.86,-17.82
109209,3,1.36,-21.59,3.93,6.25
109208,2,18.35,23.70,1.36,-21.59
109207,1,6.60,-22.32,18.35,23.70


In [48]:
for r in xy.iterrows():
  print(r[1]['shot_nbr'])
  delta_x = (r[1]['next_loc_x'] - r[1]['loc_x'])
  delta_y = (r[1]['next_loc_y'] - r[1]['loc_y'])
  print(np.arctan(delta_x/delta_y)/(np.pi/2)*90)

4.0
-24.422429850173877
3.0
5.2742099033880425
2.0
20.562976887326887
1.0
14.322968664278335


In [46]:
get_angle_seq('R26079', shot_rally_df)

,0
109207,14.322969
109208,20.562977
109209,5.274210
109210,-24.422430


In [18]:
rally_id = 'R26079'
my_df = shot_df[shot_df['rally_id'] == rally_id][['shot_nbr', 'loc_x', 'loc_y']]

In [20]:
my_df.sort_values(by='shot_nbr')

,shot_nbr,loc_x,loc_y
2,1,13.40,22.32
0,2,18.35,23.70
303831,3,18.64,21.59
119398,4,3.93,6.25
